# SingBERT Distillation — NS Commitment Stance Axis (Stage 5b — v1, Single-Head)

Distils gpt-4.1-mini commitment labels onto `zanelim/singbert-large-sg` for the **stance axis only**.
Single-head architecture — one encoder, one 3-class head.

- **Axis:** stance — supportive / critical / neutral (institutional opinion on NS as a policy)

## Training data — two sources combined
1. `commitment_llm_enrich_queue.csv` — `llm_stance` column (15k enriched chunks)
2. `commitment_llm_annual.csv` — 10,076-row validated dataset (κ=0.752) with relabelled columns:
   old `committed` → `supportive`, old `critical` → `critical`, old `neutral` → `neutral`
   This gives the stance head real minority signal it otherwise won't get.

## Design (lessons from Stage 5a and dual-head v1)
- **Split BEFORE any replication** — replicate-then-split leaks duplicates into val
- **Human rows → evaluation ONLY, LLM rows → train ONLY** — mixing gave fake val kappa=1.0
- **No class weights** — use row replication instead (supportive × 10, critical × 10, neutral × 1)
- Replicate AFTER splitting, never before
- LR=2e-5, 4 epochs max, best model by val kappa
- `save_total_limit=1` — BERT-large checkpoints are ~1.3 GB; Kaggle disk is 20 GB

## Gate
- Test kappa ≥ 0.45 AND critical recall ≥ 0.40 AND supportive recall ≥ 0.30

## Kaggle setup
- Attach dataset containing `commitment_llm_enrich_queue.csv`, `commitment_llm_annual.csv`, and `commitment_testset_queue.csv`
- GPU: T4 (BERT-large fits batch 16 at max_len 256). Expected runtime ~30–45 min for 4 epochs.

## Note on the annual dataset label mapping

The annual sample (`commitment_llm_annual.csv`) was validated at κ=0.752 on the old
committed/critical/neutral scheme. The mapping committed→supportive, critical→critical is
conceptually clean. Neutral maps to neutral.

Rationale: the original `committed` label in the annual set captured the same construct as
`supportive` stance — a soldier who expressed that NS is good/worthwhile/necessary. The
`critical` label maps directly. Neutral maps directly. The mapping is a pure rename.

In [ ]:
# Upgrade transformers + peft together to avoid EncoderDecoderCache import mismatch.
# Kaggle's pre-installed peft requires transformers>=4.43.0; pinning breaks it. NO version pins.
!pip install -q -U transformers peft datasets scikit-learn

In [ ]:
import glob
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, cohen_kappa_score,
    classification_report, confusion_matrix,
    precision_recall_fscore_support,
)
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
)

# ── Reproducibility ────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Auto-discover input files (works regardless of dataset slug name) ──────
def find_input_file(*filenames: str) -> Path:
    """Return the first match for any of the candidate filenames.

    Searches /kaggle/input recursively, then the local working dir
    (useful for local testing).
    """
    for filename in filenames:
        matches = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
        if matches:
            return Path(matches[0])
        local = Path(filename)
        if local.exists():
            return local
    raise FileNotFoundError(
        f"None of {filenames} found under /kaggle/input/ or cwd.\n"
        f"Attach the dataset containing the labelled enrich CSV via Add Data."
    )

ENRICH_PATH = find_input_file("commitment_llm_enrich_queue.csv", "commitment_llm_enrich_labelled.csv")
ANNUAL_PATH = find_input_file("commitment_llm_annual.csv")
TEST_PATH   = find_input_file("commitment_testset_queue.csv")
print(f"Enrich (LLM labels) : {ENRICH_PATH}")
print(f"Annual (validated)  : {ANNUAL_PATH}")
print(f"Test   (human)      : {TEST_PATH}")

OUTPUT_DIR = Path("/kaggle/working/singbert_ns_stance")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Model config ───────────────────────────────────────────────────────────
MODEL_NAME = "zanelim/singbert-large-sg"
MAX_LEN    = 256
BATCH_SIZE = 16      # BERT-large fits 16 on T4 16GB
GRAD_ACCUM = 2       # effective batch = 32
LR         = 2e-5
EPOCHS     = 4
VAL_FRAC   = 0.10

# ── Stance label encoding ──────────────────────────────────────────────────
LABEL2ID = {"supportive": 0, "critical": 1, "neutral": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS = 3
LABEL_ORDER = ["supportive", "critical", "neutral"]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nDevice : {DEVICE}")
print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Load enrich data — llm_stance column ─────────────────────────────────
enrich_raw = pd.read_csv(ENRICH_PATH)
print(f"Raw enrich rows: {len(enrich_raw):,}")

required_cols = {"chunk_id", "text", "llm_stance"}
missing = required_cols - set(enrich_raw.columns)
assert not missing, (
    f"Enrich CSV missing columns {missing}. Got: {list(enrich_raw.columns)}."
)

enrich_raw["llm_stance"] = enrich_raw["llm_stance"].astype(str).str.strip().str.lower()
enrich_raw = enrich_raw.dropna(subset=["text"])
enrich_raw["text"] = enrich_raw["text"].astype(str).str.strip()
enrich_raw = enrich_raw[enrich_raw["text"].str.len() > 0]

valid_mask = enrich_raw["llm_stance"].isin(LABEL2ID)
bad_labels = sorted(set(enrich_raw["llm_stance"]) - set(LABEL2ID) - {"", "nan", "none"})
if bad_labels:
    print(f"WARNING — dropping rows with unexpected stance labels: {bad_labels}")

enrich_df = enrich_raw[valid_mask].drop_duplicates("chunk_id").reset_index(drop=True)
enrich_df = enrich_df[["chunk_id", "text", "llm_stance"]].rename(columns={"llm_stance": "stance_label"})
enrich_df["source"] = "enrich"

print(f"Enrich labelled rows: {len(enrich_df):,}")
print(f"Enrich stance distribution:")
print(enrich_df["stance_label"].value_counts())

In [ ]:
# ── Load annual validated dataset — remap labels to stance scheme ─────────
# Original columns: committed → supportive, critical → critical, neutral → neutral
# κ=0.752 on original scheme; mapping is conceptually clean (see header markdown)
annual_raw = pd.read_csv(ANNUAL_PATH)
print(f"Raw annual rows: {len(annual_raw):,}")
print(f"Annual columns: {list(annual_raw.columns)}")

# Detect the label column — may be 'label', 'llm_label', 'committed', etc.
# Try common candidates
annual_label_col = None
for candidate in ["label", "llm_buyin", "llm_label", "committed", "stance"]:
    if candidate in annual_raw.columns:
        annual_label_col = candidate
        break
assert annual_label_col is not None, (
    f"Cannot find label column in annual CSV. Got columns: {list(annual_raw.columns)}. "
    f"Expected one of: label, llm_buyin, llm_label, committed, stance."
)
print(f"Using annual label column: '{annual_label_col}'")

# Clean and remap
annual_raw["_raw_label"] = annual_raw[annual_label_col].astype(str).str.strip().str.lower()
annual_raw = annual_raw.dropna(subset=["text"])
annual_raw["text"] = annual_raw["text"].astype(str).str.strip()
annual_raw = annual_raw[annual_raw["text"].str.len() > 0]

# Label remapping: old committed → supportive, critical → critical, neutral → neutral
ANNUAL_REMAP = {
    "committed" : "supportive",
    "supportive": "supportive",
    "critical"  : "critical",
    "neutral"   : "neutral",
}
annual_raw["stance_label"] = annual_raw["_raw_label"].map(ANNUAL_REMAP)
bad_annual = sorted(set(annual_raw["_raw_label"]) - set(ANNUAL_REMAP))
if bad_annual:
    print(f"WARNING — annual rows with unmapped labels (dropping): {bad_annual}")

annual_df = annual_raw.dropna(subset=["stance_label"]).copy()
annual_df = annual_df[["chunk_id", "text", "stance_label"]]
annual_df["source"] = "annual"

print(f"Annual labelled rows (after remap): {len(annual_df):,}")
print(f"Annual stance distribution (remapped):")
print(annual_df["stance_label"].value_counts())

In [ ]:
# ── Combine enrich + annual (dedupe on chunk_id — annual wins on conflict) ─
# Annual rows are validated at κ=0.752 so they take priority when chunk_ids overlap.
combined = pd.concat([enrich_df, annual_df], ignore_index=True)
n_before = len(combined)
# keep_first: annual is appended last → we want annual to win → reverse and drop_duplicates then reverse back
combined_deduped = (
    combined[::-1]
    .drop_duplicates(subset="chunk_id", keep="first")
    [::-1]
    .reset_index(drop=True)
)
n_deduped = n_before - len(combined_deduped)
print(f"Combined pool: {len(combined_deduped):,} rows ({n_deduped:,} chunk_id duplicates resolved, annual wins)")
print(f"\nOverall stance distribution:")
print(combined_deduped["stance_label"].value_counts())
print(f"\nBy source:")
print(pd.crosstab(combined_deduped["source"], combined_deduped["stance_label"]))

llm_df = combined_deduped.copy()
llm_df["label_id"] = llm_df["stance_label"].map(LABEL2ID)

In [ ]:
# ── Load human test set (held-out — NEVER trained on) ──────────────────────
test_raw = pd.read_csv(TEST_PATH)

assert "human_stance" in test_raw.columns, (
    f"commitment_testset_queue.csv has no 'human_stance' column for stance axis. "
    f"Got: {list(test_raw.columns)}"
)

test_raw["human_stance"] = test_raw["human_stance"].astype(str).str.strip().str.lower()
valid_test = test_raw["human_stance"].isin(LABEL2ID)
test_df = test_raw[valid_test].drop_duplicates("chunk_id").reset_index(drop=True)

assert len(test_df) >= 50, (
    f"Only {len(test_df)} labelled test rows — expected the completed 200-row test set."
)
print(f"Human test rows (stance): {len(test_df)} / {len(test_raw)}")
print(f"\nHuman stance distribution:")
print(test_df["human_stance"].value_counts())

# ── Leakage guard: human test chunk_ids must NOT appear in training pool ────
leak = set(llm_df["chunk_id"]) & set(test_df["chunk_id"])
if leak:
    print(f"WARNING — {len(leak)} chunk_ids overlap between train pool and test set; "
          f"removing them from the TRAINING pool.")
    llm_df = llm_df[~llm_df["chunk_id"].isin(leak)].reset_index(drop=True)
assert not (set(llm_df["chunk_id"]) & set(test_df["chunk_id"])), "Leakage guard failed"
print("\nLeakage check passed — zero chunk_id overlap between train pool and human test set.")

In [ ]:
# ── Train/val split — SPLIT FIRST, then replicate ─────────────────────────
# Split BEFORE replication — never after (replicate-then-split leaks duplicates into val)
# Human rows → val ONLY. LLM rows → train ONLY. No mixing.
train_df, val_df = train_test_split(
    llm_df,
    test_size=VAL_FRAC,
    random_state=SEED,
    stratify=llm_df["label_id"],
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print(f"Before replication:")
print(f"  Train: {len(train_df):,}  Val: {len(val_df):,}")
print(f"  Train dist: {train_df['stance_label'].value_counts().to_dict()}")

# ── Replication: supportive × 10, critical × 10, neutral × 1 ──────────────
# 10× replication for both — targeting 70% recall on minority classes.
# Split MUST precede this step.
REPLICATE_SUPPORTIVE = 10
REPLICATE_CRITICAL   = 10

train_supportive = train_df[train_df["stance_label"] == "supportive"]
train_critical   = train_df[train_df["stance_label"] == "critical"]
train_neutral    = train_df[train_df["stance_label"] == "neutral"]

replicated_parts = [train_neutral]  # neutral × 1
for _ in range(REPLICATE_SUPPORTIVE):
    replicated_parts.append(train_supportive)
for _ in range(REPLICATE_CRITICAL):
    replicated_parts.append(train_critical)

train_df = pd.concat(replicated_parts, ignore_index=True).sample(
    frac=1, random_state=SEED
).reset_index(drop=True)

print(f"\nAfter replication (supportive × {REPLICATE_SUPPORTIVE}, critical × {REPLICATE_CRITICAL}, neutral × 1):")
print(f"  Train: {len(train_df):,}  Val: {len(val_df):,}")
print(f"  Train dist: {train_df['stance_label'].value_counts().to_dict()}")
print(f"  Val dist:   {val_df['stance_label'].value_counts().to_dict()}")
print(f"  Test:  {len(test_df)} rows (human labels — final gate, untouched until the end)")

In [ ]:
# ── Tokenizer + dataset ───────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Tokenising train ({len(train_df):,}) ...")
train_enc = tokenizer(train_df["text"].tolist(), truncation=True, max_length=MAX_LEN, padding=False)
print(f"Tokenising val   ({len(val_df):,}) ...")
val_enc   = tokenizer(val_df["text"].tolist(),   truncation=True, max_length=MAX_LEN, padding=False)
print("Done.")


class StanceDataset(Dataset):
    def __init__(self, encodings, label_ids):
        self.encodings = encodings
        self.labels    = label_ids

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)


train_dataset = StanceDataset(train_enc, train_df["label_id"].tolist())
val_dataset   = StanceDataset(val_enc,   val_df["label_id"].tolist())
print(f"Train dataset : {len(train_dataset):,}")
print(f"Val dataset   : {len(val_dataset):,}")

In [ ]:
# ── Model ─────────────────────────────────────────────────────────────────
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
)
model.to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: {MODEL_NAME}")
print(f"Total params    : {total_params:,}")
print(f"Trainable params: {trainable:,}")
print(f"Labels: {LABEL2ID}")

In [ ]:
# ── Metrics + Trainer ─────────────────────────────────────────────────────

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    kappa = cohen_kappa_score(labels, preds)
    acc   = accuracy_score(labels, preds)
    return {"kappa": kappa, "accuracy": acc}


training_args = TrainingArguments(
    output_dir                  = str(OUTPUT_DIR),
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE * 2,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LR,
    weight_decay                = 0.01,
    warmup_ratio                = 0.06,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    save_total_limit            = 1,          # BERT-large ckpt ~1.3 GB; Kaggle disk 20 GB
    load_best_model_at_end      = True,
    metric_for_best_model       = "kappa",
    greater_is_better           = True,
    fp16                        = torch.cuda.is_available(),
    dataloader_num_workers      = 2,
    logging_steps               = 50,
    report_to                   = "none",
    seed                        = SEED,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Version-aware tokenizer kwarg: transformers>=4.46 → processing_class=, else tokenizer=
import transformers
_trainer_kwargs = dict(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = val_dataset,
    data_collator   = data_collator,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)],
)
tv = tuple(int(x) for x in transformers.__version__.split(".")[:2])
if tv >= (4, 46):
    _trainer_kwargs["processing_class"] = tokenizer
else:
    _trainer_kwargs["tokenizer"] = tokenizer

# Plain Trainer — no class weights (replication handles balance at data level)
trainer = Trainer(**_trainer_kwargs)

steps_per_epoch = max(1, len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM))
print("Trainer configured (no class weights — replication corrects neutral-heavy bias).")
print(f"Best model selected by: kappa (val — LLM-labelled rows)")
print(f"~{steps_per_epoch} optimizer steps/epoch x {EPOCHS} epochs max")

In [ ]:
# ── Train ──────────────────────────────────────────────────────────────────
print("Starting training ...")
train_result = trainer.train()
print("\nTraining complete.")
print(f"  Total steps   : {train_result.global_step}")
print(f"  Training loss : {train_result.training_loss:.4f}")

metrics = trainer.evaluate()
print(f"\nBest val (LLM-label) metrics:")
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

In [ ]:
# ── Human test-set evaluation — the Stage 5b gate (stance axis) ───────────
test_enc = tokenizer(
    test_df["text"].astype(str).tolist(),
    truncation=True, max_length=MAX_LEN, padding=False,
)
test_label_ids = test_df["human_stance"].map(LABEL2ID).tolist()
test_dataset   = StanceDataset(test_enc, test_label_ids)

test_out   = trainer.predict(test_dataset)
logits_np  = test_out.predictions
probs      = torch.softmax(torch.tensor(logits_np, dtype=torch.float32), dim=-1).numpy()
preds      = np.argmax(logits_np, axis=-1)
preds_named = [ID2LABEL[p] for p in preds]
true_named  = test_df["human_stance"].tolist()

acc   = accuracy_score(test_label_ids, preds)
kappa = cohen_kappa_score(test_label_ids, preds)

print("=" * 72)
print("  Human Test-Set Evaluation — SingBERT stance distill v1")
print("=" * 72)
print(f"  Rows evaluated : {len(test_label_ids)}")
print(f"  Accuracy       : {acc:.3f}  ({acc*100:.1f}%)")
print(f"  Cohen's Kappa  : {kappa:.3f}")
print()
print(classification_report(true_named, preds_named, labels=LABEL_ORDER, digits=3, zero_division=0))
print("  Confusion matrix (rows=human, cols=predicted):")
cm = confusion_matrix(true_named, preds_named, labels=LABEL_ORDER)
print(pd.DataFrame(cm, index=[f"h_{c}" for c in LABEL_ORDER],
                       columns=[f"p_{c}" for c in LABEL_ORDER]))

# ── Per-class recall for gate check ───────────────────────────────────────
p_vals, r_vals, f_vals, _ = precision_recall_fscore_support(
    true_named, preds_named, labels=LABEL_ORDER, average=None, zero_division=0
)
recall_critical   = r_vals[LABEL_ORDER.index("critical")]
recall_supportive = r_vals[LABEL_ORDER.index("supportive")]

# ── Gate check ─────────────────────────────────────────────────────────────
GATE_KAPPA          = 0.50
GATE_CRITICAL_REC   = 0.50
GATE_SUPPORTIVE_REC = 0.55

kappa_pass     = kappa            >= GATE_KAPPA
critical_pass  = recall_critical   >= GATE_CRITICAL_REC
support_pass   = recall_supportive >= GATE_SUPPORTIVE_REC
gate_pass      = kappa_pass and critical_pass and support_pass

print(f"\n  --- GATE ---")
print(f"  kappa >= {GATE_KAPPA}           : {kappa:.3f}   {'PASS' if kappa_pass else 'FAIL'}")
print(f"  critical recall >= {GATE_CRITICAL_REC}    : {recall_critical:.3f}   {'PASS' if critical_pass else 'FAIL'}")
print(f"  supportive recall >= {GATE_SUPPORTIVE_REC}  : {recall_supportive:.3f}   {'PASS' if support_pass else 'FAIL'}")
print(f"  Overall gate: {'PASS — proceed to Stage 6' if gate_pass else 'FAIL — apply lexicon override + threshold tuning before proceeding'}")
print("=" * 72)

In [ ]:
# ── Save test predictions + summary ────────────────────────────────────────
eval_df = test_df.copy()
eval_df["singbert_stance"]         = preds_named
eval_df["prob_stance_supportive"]   = probs[:, LABEL2ID["supportive"]]
eval_df["prob_stance_critical"]     = probs[:, LABEL2ID["critical"]]
eval_df["prob_stance_neutral"]      = probs[:, LABEL2ID["neutral"]]
eval_df["stance_correct"]           = eval_df["human_stance"] == eval_df["singbert_stance"]

eval_path = OUTPUT_DIR / "stance_testset_eval.csv"
eval_df.to_csv(eval_path, index=False)
print(f"Saved -> {eval_path}")

summary = {
    "model"          : MODEL_NAME,
    "axis"           : "stance",
    "architecture"   : "single-head AutoModelForSequenceClassification (3-class)",
    "label2id"       : LABEL2ID,
    "training_sources": ["commitment_llm_enrich_queue.csv (llm_stance)",
                          "commitment_llm_annual.csv (committed→supportive, critical→critical, neutral→neutral)"],
    "train_rows"     : len(train_dataset),
    "val_rows"       : len(val_dataset),
    "test_rows"      : len(test_label_ids),
    "accuracy"       : round(acc,   4),
    "kappa"          : round(kappa, 4),
    "critical_recall"  : round(float(recall_critical),   4),
    "supportive_recall": round(float(recall_supportive), 4),
    "gate_pass"      : bool(gate_pass),
    "max_len"        : MAX_LEN,
    "lr"             : LR,
    "epochs_max"     : EPOCHS,
    "replication"    : f"supportive × {REPLICATE_SUPPORTIVE}, critical × {REPLICATE_CRITICAL}, neutral × 1",
}
with open(OUTPUT_DIR / "eval_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

In [ ]:
# ── Save model + tokenizer + id2label ─────────────────────────────────────
import shutil

model_out = OUTPUT_DIR / "best_model"
trainer.save_model(str(model_out))
tokenizer.save_pretrained(str(model_out))
print(f"Model saved -> {model_out}")

# Save id2label for the infer notebook to validate against
id2label_path = model_out / "id2label.json"
with open(id2label_path, "w") as f:
    json.dump({"axis": "stance", "id2label": ID2LABEL, "label2id": LABEL2ID}, f, indent=2)
print(f"Saved id2label -> {id2label_path}")

for fp in sorted(model_out.iterdir()):
    print(f"  {fp.name:<40} {fp.stat().st_size / 1e6:>8.1f} MB")

zip_path = str(OUTPUT_DIR.parent / "singbert_ns_stance")
shutil.make_archive(zip_path, "zip", str(model_out))
zip_size = Path(zip_path + ".zip").stat().st_size / 1e6
print(f"\nZipped -> {zip_path}.zip  ({zip_size:.0f} MB)")
print()
print("Next steps:")
print("  1. Check stance gate thresholds above (kappa ≥ 0.45, critical recall ≥ 0.40, supportive recall ≥ 0.30)")
print("  2. Upload singbert_ns_stance.zip as a Kaggle dataset")
print("  3. Also run kaggle_distill_buyin_v1.ipynb for the buyin axis")
print("  4. Then run kaggle_infer_commitment_v1.ipynb for full 737k dual-axis inference")

## Gate thresholds (v5 — updated for enriched training data)

Pass criteria before proceeding to inference:
- **kappa ≥ 0.50**
- **critical recall ≥ 0.50** — model alone target; lexicon override + threshold tuning push to 0.70
- **supportive recall ≥ 0.55**

If gate fails, apply lexicon override and threshold tuning before re-evaluating.

Expected ranges after enriched retrain:
- Critical recall:   0.55–0.65 (model alone) → 0.65–0.72 (+ lexicon + threshold)
- Supportive recall: 0.60–0.70